<a href="https://colab.research.google.com/github/Innovatewithapple/CNNProjects/blob/main/PytorchPreTrainedCNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
import torch.nn as nn

from torchvision import datasets,transforms
from torch.utils.data import DataLoader
import torch.optim as optimizer

import timm
import os
from google.colab import userdata

In [ ]:
# 'tf_' refers to the official weights ported from the Google/TensorFlow team
model = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0)


In [ ]:
# 1. Get the config from your model (run this once)
config = model.default_cfg
print(config)

{'url': 'https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-effv2-weights/tf_efficientnetv2_s_21ft1k-d7dafa41.pth', 'hf_hub_id': 'timm/tf_efficientnetv2_s.in21k_ft_in1k', 'architecture': 'tf_efficientnetv2_s', 'tag': 'in21k_ft_in1k', 'custom_load': False, 'input_size': (3, 300, 300), 'test_input_size': (3, 384, 384), 'fixed_input_size': False, 'interpolation': 'bicubic', 'crop_pct': 1.0, 'crop_mode': 'center', 'mean': (0.5, 0.5, 0.5), 'std': (0.5, 0.5, 0.5), 'num_classes': 1000, 'pool_size': (10, 10), 'first_conv': 'conv_stem', 'classifier': 'classifier', 'license': 'apache-2.0'}


In [ ]:
os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('key and username activated!')

key and username activated!


In [ ]:
!kaggle datasets download -d xhlulu/140k-real-and-fake-faces

Dataset URL: https://www.kaggle.com/datasets/xhlulu/140k-real-and-fake-faces
License(s): other
140k-real-and-fake-faces.zip: Skipping, found more recently modified local copy (use --force to force download)


In [ ]:
!unzip -q 140k-real-and-fake-faces.zip -d ./dataset_folder

replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/00276TOPP4.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/008BYSE725.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/009ZTJ3621.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/00F8LKY6JC.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: a
error:  invalid response [a]
replace ./dataset_folder/real_vs_fake/real-vs-fake/test/fake/00F8LKY6JC.jpg? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [ ]:
train_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/train'
test_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/test'
validation_dir = '/content/dataset_folder/real_vs_fake/real-vs-fake/valid'

In [ ]:
train_transfom = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # We grab the 'mean' and 'std' keys directly from the model's brain!
    transforms.Normalize(mean=config['mean'],
                         std=config['std'])
])

val_transfom = transforms.Compose([
    transforms.Resize((384,384)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    # We grab the 'mean' and 'std' keys directly from the model's brain!
    transforms.Normalize(mean=config['mean'],
                         std=config['std'])
])

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)
pin_memory = (True if torch.cuda.is_available() else False)

cuda


In [ ]:
train_dataset = datasets.ImageFolder(root=train_dir,transform=train_transfom)
val_dataset = datasets.ImageFolder(root=validation_dir,transform=val_transfom)

train_loader = DataLoader(dataset=train_dataset,batch_size=32,shuffle=True,num_workers=2,pin_memory=pin_memory)
val_loader = DataLoader(dataset=val_dataset,batch_size=32,num_workers=2,pin_memory=pin_memory)

In [ ]:
#Model initialisation
# 2. FREEZE THE BRAIN: Lock the ImageNet knowledge
for params in model.parameters():
  params.requires_grad = False

# 3. THE HEAD: This is your custom part (exactly like your image)
my_custom_head = nn.Sequential(
    nn.LazyLinear(128),
    nn.ReLU(),
    nn.Linear(128,1)
)

#Assenmble
assembled_Model = nn.Sequential(
    model,
    my_custom_head
)

assembled_Model = assembled_Model.to(device)

In [ ]:
dummyImg = torch.randn(1,3,384,384).to(device)
assembled_Model(dummyImg)
print('✅ Model Assembled! Ready for Epochs.')

✅ Model Assembled! Ready for Epochs.


In [ ]:
loss_fn = nn.BCEWithLogitsLoss()
optimizer = optimizer.Adam(assembled_Model.parameters(),lr=0.001)

In [ ]:
#Training
epochs = 10

for epoch in range(epochs):
  model.train()
  train_loss = 0
  train_correct = 0
  train_total = 0

  for i, (images,labels) in enumerate(train_loader):
    images = images.to(device)
    labels = labels.float().unsqueeze(1).to(device)

    optimizer.zero_grad()

    output = assembled_Model(images)
    loss = loss_fn(output,labels)

    loss.backward()
    optimizer.step()

    if i % 100 == 0:
        print(f"Batch {i} - Loss: {loss.item():.4f}", end='\r')

    train_loss += loss.item()
    preds = (torch.sigmoid(output) > 0.5).int()
    train_correct += (preds.squeeze() == labels.squeeze()).sum().item()

    train_total += labels.size(0)

  training_accuracy = train_correct / train_total

  #evlauation
  model.eval()
  val_loss = 0
  val_correct = 0
  val_total = 0

  with torch.no_grad():
    for images,labels in val_loader:
     images = images.to(device)
     labels = labels.float().unsqueeze(1).to(device)

     output = assembled_Model(images)
     loss = loss_fn(output,labels)

     val_loss += loss.item()
     preds = (torch.sigmoid(output) > 0.5).int()
     val_correct += (preds.squeeze() == labels.squeeze()).sum().item()

     val_total += labels.size(0)

  validation_accuracy = val_correct / val_total

      # Calculate averages for the whole epoch
  avg_train_loss = train_loss / len(train_loader)
  avg_val_loss = val_loss / len(val_loader)

  # The "Professional Dashboard" Print
  print(f"\n🚀 Epoch [{epoch+1}/{epochs}]")
  print(f"-------------------------------")
  print(f"TRAIN | Loss: {avg_train_loss:.4f} | Acc: {training_accuracy:.4f}")
  print(f"VALID | Loss: {avg_val_loss:.4f} | Acc: {validation_accuracy:.4f}")
  print(f"-------------------------------\n")



🚀 Epoch [1/10]
-------------------------------
TRAIN | Loss: 0.2340 | Acc: 0.9023
VALID | Loss: 0.1677 | Acc: 0.9324
-------------------------------


🚀 Epoch [2/10]
-------------------------------
TRAIN | Loss: 0.1603 | Acc: 0.9363
VALID | Loss: 0.1577 | Acc: 0.9351
-------------------------------


🚀 Epoch [3/10]
-------------------------------
TRAIN | Loss: 0.1320 | Acc: 0.9485
VALID | Loss: 0.0997 | Acc: 0.9616
-------------------------------



In [ ]:
# --- RUN THIS AFTER PHASE 1 IS FINISHED ---

# 1. UNFREEZE THE BRAIN
# We set this to True so the 'Expert' layers can start adjusting
for param in model.parameters():
    param.requires_grad = True

# 2. THE "SURGERY": Selective Freezing
# EfficientNetV2-S has about 300+ layers.
# We freeze the first 250 (the "Eyes" that see lines/colors)
# and leave the last 50+ (the "Deep Thoughts" that see skin/pores) awake.
all_params = list(model.parameters())
for param in all_params[:250]:
    param.requires_grad = False

# 3. RE-DEFINE THE OPTIMIZER
# We MUST include both your custom head AND the brain in the optimizer
optimizer = torch.optim.Adam([
    {'params': model.parameters()},      # The now-awake brain
    {'params': my_custom_head.parameters()} # Your custom dense layers (always awake)
], lr=0.00001) # TINY speed for fine-tuning

print("🔓 Surgery Complete. The 'Deep Brain' and 'Custom Head' are now learning together!")


In [3]:
for i, (name, layer) in enumerate(model.named_modules()):
    print(f"Layer ID: {i} | Name: {name}")

Layer ID: 0 | Name: 
Layer ID: 1 | Name: conv_stem
Layer ID: 2 | Name: bn1
Layer ID: 3 | Name: bn1.drop
Layer ID: 4 | Name: bn1.act
Layer ID: 5 | Name: blocks
Layer ID: 6 | Name: blocks.0
Layer ID: 7 | Name: blocks.0.0
Layer ID: 8 | Name: blocks.0.0.conv
Layer ID: 9 | Name: blocks.0.0.bn1
Layer ID: 10 | Name: blocks.0.0.bn1.drop
Layer ID: 11 | Name: blocks.0.0.bn1.act
Layer ID: 12 | Name: blocks.0.0.aa
Layer ID: 13 | Name: blocks.0.0.drop_path
Layer ID: 14 | Name: blocks.0.1
Layer ID: 15 | Name: blocks.0.1.conv
Layer ID: 16 | Name: blocks.0.1.bn1
Layer ID: 17 | Name: blocks.0.1.bn1.drop
Layer ID: 18 | Name: blocks.0.1.bn1.act
Layer ID: 19 | Name: blocks.0.1.aa
Layer ID: 20 | Name: blocks.0.1.drop_path
Layer ID: 21 | Name: blocks.1
Layer ID: 22 | Name: blocks.1.0
Layer ID: 23 | Name: blocks.1.0.conv_exp
Layer ID: 24 | Name: blocks.1.0.bn1
Layer ID: 25 | Name: blocks.1.0.bn1.drop
Layer ID: 26 | Name: blocks.1.0.bn1.act
Layer ID: 27 | Name: blocks.1.0.aa
Layer ID: 28 | Name: blocks.1.0.se